In [1]:
import polars as pl
import pandas as pd
import xarray as xr
import numpy as np
from jax import numpy as jnp, jit
from itertools import product, combinations, permutations
from bidict import bidict

In [2]:
from summer3.proto import CompartmentMap, Stratification

In [3]:
age_strat = Stratification("age", ["infant", "child", "adult", "older"])

cm = CompartmentMap.new(age_strat)

In [4]:
disease_state = cm.stratify(Stratification("disease_state", ["S","I","R"]))

In [5]:
locx = np.arange(8)
locy = np.arange(8)
loc_strata = [f"loc_{i}" for i,s in enumerate(product(locx, locy))]

In [6]:
loc_strat = cm.stratify(Stratification("loc", loc_strata))

In [6]:
data = cm.zeros()

In [9]:
ma = data.as_managed_array()

In [10]:
from summer3.utils import get_unique_keyname

In [ ]:
# Unifying access across
# 1d vs Nd
# Compartment


class PAccessExpr:
    def __init__(self, pl_expr, pa):
        self.pl_expr = pl_expr
        self.pa = pa

    def __and__(self, other):
        return PAccessExpr(self.pl_expr & other.pl_expr, self.pa)
    
    def __invert__(self):
        return PAccessExpr(~self.pl_expr, self.pa)
    
    def __repr__(self):
        return f"PAccessExpr[{self.pa}] {self.pl_expr}"

class PropertyAccessor:
    # prop_key is probably a Stratification for our first example
    def __init__(self, prop_key, ptable):
        self.prop_key = prop_key
        self._ptable = ptable

        self._uname = uname = ptable.su[prop_key]
        self._pik_map = ptable.uname_propidxkey_map[uname]

    def __repr__(self):
        return f"PropertyAccessor[{self.prop_key}]"

    def __map_args__(self, args):
        try:
            iargs = self._pik_map.inverse[args]
        except:
            iargs = [self._pik_map.inverse[a] for a in args]

        return iargs

    def is_in(self, args):
        if isinstance(args, str):
            args = [args]
        iargs = self.__map_args__(args)
        pl_expr = pl.col(self._uname).is_in(iargs)
        return PAccessExpr(pl_expr, self)


In [53]:
class PropertyTable:
    def __init__(self, uname_strat_map, uname_propidxkey_map, prop_table):
        self.uname_strat_map = uname_strat_map
        self.uname_propidxkey_map = uname_propidxkey_map
        self.table = prop_table

        self.us = self.uname_strat_map
        self.su = self.uname_strat_map.inverse

        self.properties = self.__build_accessors__()

    def __build_accessors__(self):
        return {k:PropertyAccessor(k, self) for k in self.uname_strat_map.inverse}

def build_property_tables(cm, compartments):
    # Stratifications do not require unique names
    # but we need unique strings for non-object supporting 
    # table keys (polars etc)
    uname_strat_map = bidict()
    uname_propidxkey_map: dict[str,bidict] = {}
    _prop_tab = {}
    for strat in cm.stratifications:
        name = strat.name
        uname = get_unique_keyname(name, uname_strat_map)
        uname_strat_map[uname] = strat
        _prop_tab[uname] = v = np.empty(len(compartments), dtype=int)
        v.fill(-1)
        uname_propidxkey_map[uname] = bidict({i:k for (i,k) in enumerate(strat.strata)})
    
    for comp_i,c in enumerate(compartments):
        for strat, stratum in c.strata:
            uname = uname_strat_map.inverse[strat]
            ik_map = uname_propidxkey_map[uname]
            prop_i = ik_map.inverse[stratum]
            _prop_tab[uname][comp_i] = prop_i

    prop_table = pl.DataFrame(_prop_tab | {"index": np.arange(len(compartments))})

    return PropertyTable(uname_strat_map, uname_propidxkey_map, prop_table)

In [54]:
prop_table = build_property_tables(cm, cm.compartments)

In [58]:
prop_table.properties[disease_state].is_in(["S","R"])

PAccessExpr[PropertyAccessor[Stratification: disease_state]] col("disease_state_0").is_in([[0, 2]])

In [19]:
def pfilter(ptable: PropertyTable, strat, op, args, inv=False):
    uname = ptable.su[strat]
    pik_map = ptable.uname_propidxkey_map[uname]
    try:
        iargs = pik_map.inverse[args]
    except:
        iargs = [pik_map.inverse[a] for a in args]
    if inv:
        return ptable.table.filter(~getattr(pl.col(uname), op)(iargs))
    else:
        return ptable.table.filter(getattr(pl.col(uname), op)(iargs))

In [20]:
col = pl.col("age_0")

In [ ]:
age_pa = PropertyAccessor(age_strat, prop_table)
disease_pa = PropertyAccessor(disease_state, prop_table)

q = age_pa.is_in(["child"]) and ~disease_pa.is_in("S")

prop_table.table.filter(q.pl_expr)

age_0,disease_state_0,index
i64,i64,i64
0,1,1
0,2,2
1,1,4
1,2,5
2,1,7
2,2,8
3,1,10
3,2,11


In [22]:
prop_table.table.filter(pl.col("age_0").is_between(1,2) & pl.col("disease_state_0").is_in([0,2]))

age_0,disease_state_0,index
i64,i64,i64
1,0,3
1,2,5
2,0,6
2,2,8


In [ ]:
def get_pmask(prop_dict, ptable: PropertyTable):
    for strat, strata in prop_dict.items():
        

In [15]:
prop_table.uname_propidxkey_map[prop_table.su[age_strat]].inverse["child"]

1

In [18]:
prop_table.table

age_0,disease_state_0,index
i64,i64,i64
0,0,0
0,1,1
0,2,2
1,0,3
1,1,4
…,…,…
2,1,7
2,2,8
3,0,9


In [17]:
ptt = pfilter(age_strat, "is_in", ["infant","child"])

[0, 1]


In [ ]:
# is_tiling_dim
# Can we reshape this array such that a new axis
# constructed from this dimension will have equal tiling
# with regards to other data

In [91]:
cdata = cm.wrap_data(jnp.arange(0,len(cm.compartments))).as_managed_array()

In [92]:
cdata.data

Array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

In [121]:
cm.query([(age_strat, ["child","older"]),(loc_strat, ["loc_0","loc_5"])])

CompartmentContainer view of 0x2531430577360:
array([Compartment :[(Stratification: age, 'child'), (Stratification: loc, 'loc_0'), (Stratification: disease_state, 'S')],
       Compartment :[(Stratification: age, 'child'), (Stratification: loc, 'loc_0'), (Stratification: disease_state, 'I')],
       Compartment :[(Stratification: age, 'child'), (Stratification: loc, 'loc_0'), (Stratification: disease_state, 'R')],
       Compartment :[(Stratification: age, 'child'), (Stratification: loc, 'loc_5'), (Stratification: disease_state, 'S')],
       Compartment :[(Stratification: age, 'child'), (Stratification: loc, 'loc_5'), (Stratification: disease_state, 'I')],
       Compartment :[(Stratification: age, 'child'), (Stratification: loc, 'loc_5'), (Stratification: disease_state, 'R')],
       Compartment :[(Stratification: age, 'older'), (Stratification: loc, 'loc_0'), (Stratification: disease_state, 'S')],
       Compartment :[(Stratification: age, 'older'), (Stratification: loc, 'loc_0'), (

In [113]:
def is_valid_tiling_dim(dim_col, ptable):
    active_col = dim_col
    nicols = ptable.columns
    nicols.remove('index')
    nicols.remove(active_col)

    ref = None

    for col_val in ptt[active_col].unique():
        comp = ptable.filter(pl.col(active_col).eq(col_val))[nicols]
        if ref is not None:
            if not comp.equals(ref):
                return False
            ref = comp
    
    return True

In [ ]:
# Folding/Unfolding
# Consider the inverse case first;
# N-properties with perfect encompassing tiling
# represented as Nd MA, with N dims each having 1 property
# 
# What transforms can we do on this that we can prove
# retain some or all of these properties?

#
#  
# 

age_0,loc_0,disease_state_0,index
i64,i64,i64,i64
0,0,0,0
0,0,1,1
0,0,2,2
0,1,0,3
0,1,1,4
…,…,…,…
3,62,1,763
3,62,2,764
3,63,0,765


In [118]:
is_valid_tiling_dim("disease_state_0", prop_table.table)

True

In [95]:
ptt["disease_state_0"].value_counts()["count"].unique()

count
u32
192


In [74]:
ptidx = prop_table.table.filter(~pl.col("age_0").gt(0))["index"]

In [77]:
cm.compartments[ptidx]

array([Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_0'), (Stratification: disease_state, 'S')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_0'), (Stratification: disease_state, 'I')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_0'), (Stratification: disease_state, 'R')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_1'), (Stratification: disease_state, 'S')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_1'), (Stratification: disease_state, 'I')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_1'), (Stratification: disease_state, 'R')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_2'), (Stratification: disease_state, 'S')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_2'), (Stratification: disease_state, 'I')],


In [33]:
list(cm.stratifications)[0].strata

('infant', 'child', 'adult', 'older')

In [ ]:
cm.stratifications

for c in data.compartments:
    c.strata

array([Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_0'), (Stratification: disease_state, 'S')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_0'), (Stratification: disease_state, 'I')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_0'), (Stratification: disease_state, 'R')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_1'), (Stratification: disease_state, 'S')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_1'), (Stratification: disease_state, 'I')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_1'), (Stratification: disease_state, 'R')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_2'), (Stratification: disease_state, 'S')],
       Compartment :[(Stratification: age, 'infant'), (Stratification: loc, 'loc_2'), (Stratification: disease_state, 'I')],
